In [13]:
import os, subprocess, sys
for pkg in ["mlxtend", "ripser", "persim", "tqdm"]:
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                            capture_output=True, text=True)
    print(f"✅ {pkg}")

✅ mlxtend
✅ ripser
✅ persim
✅ tqdm


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  LOTTERY PREDICTOR V3
#  Fixes: ESN no-repeat enforcement | DMD centering | O-Info order 24
#  New:   10x feature expansion (gaps, primes, fib, hot/cold, etc.)
# ══════════════════════════════════════════════════════════════════
import warnings; warnings.filterwarnings("ignore")
import itertools, sqlite3
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"
print(f"✅ Device: {DEVICE}")

try:
    import ripser
    HAS_RIPSER = True
    print("✅ ripser available — running TDA")
except ImportError:
    HAS_RIPSER = False
    print("⚠️  ripser not found — TDA skipped")

PRIMES    = {2,3,5,7,11,13,17,19,23,29,31,37,41,43,47,53,59,61,67,71}
FIBS      = {1,2,3,5,8,13,21,34,55,89}

# ───────────────────────────────────────────────────────────────
#  1. DATA FETCH
# ───────────────────────────────────────────────────────────────
def fetch_lottery(name, url, spec_col, spec_in_string=False):
    print(f"🌐 Fetching {name}...")
    df = pd.read_csv(url)
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    for c in list(df.columns):
        if 'draw' in c and 'date' in c:
            df.rename(columns={c: 'draw_date'}, inplace=True); break
    for c in list(df.columns):
        if c in ('winning_numbers', 'numbers'):
            df.rename(columns={c: 'winning_numbers'}, inplace=True); break
    for c in list(df.columns):
        if c in ('mega_ball', 'megaball'):
            df.rename(columns={c: 'mb'}, inplace=True); break
    df['draw_date'] = pd.to_datetime(df['draw_date'], errors='coerce')
    df = df.dropna(subset=['draw_date']).sort_values('draw_date').reset_index(drop=True)
    sp = df['winning_numbers'].astype(str).str.split(expand=True)
    for i, col in enumerate(['n1','n2','n3','n4','n5']):
        df[col] = pd.to_numeric(sp[i], errors='coerce')
    if spec_in_string:
        df[spec_col] = pd.to_numeric(sp[5], errors='coerce')
    else:
        df[spec_col] = pd.to_numeric(df.get(spec_col, pd.Series(dtype=float)), errors='coerce')
    df = df.dropna(subset=['n1','n2','n3','n4','n5', spec_col]).reset_index(drop=True)
    latest = df.iloc[-1]
    nums = '-'.join(str(int(latest[c])) for c in ['n1','n2','n3','n4','n5'])
    print(f"✅ {name}: {len(df)} draws | Latest: {latest['draw_date'].date()} | {nums} {spec_col.upper()}:{int(latest[spec_col])}")
    return df

pb_raw = fetch_lottery("Powerball",     "https://data.ny.gov/api/views/d6yy-54nr/rows.csv?accessType=DOWNLOAD", spec_col='pb', spec_in_string=True)
mm_raw = fetch_lottery("Mega Millions", "https://data.ny.gov/api/views/5xaw-6ayf/rows.csv?accessType=DOWNLOAD",  spec_col='mb', spec_in_string=False)

# ───────────────────────────────────────────────────────────────
#  2. FEATURE ENGINEERING  (10x expanded)
# ───────────────────────────────────────────────────────────────
def build_features(df, spec_col):
    df = df.copy()
    df['draw_idx']     = np.arange(len(df))
    df['dow']          = df['draw_date'].dt.dayofweek
    df['dow_name']     = df['draw_date'].dt.day_name()
    df['month']        = df['draw_date'].dt.month
    df['year']         = df['draw_date'].dt.year
    df['week_of_year'] = df['draw_date'].dt.isocalendar().week.astype(int)

    # ── mod patterns (3,5,7,11,13) ──
    for col in ['n1','n2','n3','n4','n5']:
        for m in [3, 5, 7, 11, 13]:
            df[f'{col}_mod{m}'] = df[col] % m

    # ── pairwise sums & diffs ──
    for a, b in [('n1','n2'),('n2','n3'),('n3','n4'),('n4','n5'),('n1','n3'),('n1','n5'),('n2','n4')]:
        df[f'{a}_{b}_sum']  = df[a] + df[b]
        df[f'{a}_{b}_diff'] = (df[a] - df[b]).abs()

    # ── consecutive gaps (sorted draw) ──
    for i in range(1, 5):
        df[f'gap_{i}_{i+1}'] = df[f'n{i+1}'] - df[f'n{i}']

    # ── draw-level stats ──
    df['draw_sum']    = df[['n1','n2','n3','n4','n5']].sum(axis=1)
    df['draw_std']    = df[['n1','n2','n3','n4','n5']].std(axis=1)
    df['draw_range']  = df['n5'] - df['n1']
    df['draw_mean']   = df[['n1','n2','n3','n4','n5']].mean(axis=1)
    df['draw_median'] = df[['n1','n2','n3','n4','n5']].median(axis=1)
    df['draw_skew']   = df[['n1','n2','n3','n4','n5']].apply(lambda r: pd.Series(r).skew(), axis=1)

    # ── even/odd & low/high counts ──
    for col in ['n1','n2','n3','n4','n5']:
        df[f'{col}_even'] = (df[col] % 2 == 0).astype(int)
    df['even_count'] = df[['n1_even','n2_even','n3_even','n4_even','n5_even']].sum(axis=1)
    df['odd_count']  = 5 - df['even_count']

    # ── prime indicators ──
    for col in ['n1','n2','n3','n4','n5']:
        df[f'{col}_is_prime'] = df[col].apply(lambda x: int(x in PRIMES))
    df['prime_count'] = df[['n1_is_prime','n2_is_prime','n3_is_prime','n4_is_prime','n5_is_prime']].sum(axis=1)

    # ── Fibonacci indicators ──
    for col in ['n1','n2','n3','n4','n5']:
        df[f'{col}_is_fib'] = df[col].apply(lambda x: int(x in FIBS))
    df['fib_count'] = df[['n1_is_fib','n2_is_fib','n3_is_fib','n4_is_fib','n5_is_fib']].sum(axis=1)

    # ── digit sums ──
    for col in ['n1','n2','n3','n4','n5']:
        df[f'{col}_digsum'] = df[col].apply(lambda x: sum(int(d) for d in str(int(x))))

    # ── quadrant counts (split ball range into 4 zones) ──
    # Powerball white: 1-69 → zones 1-17,18-34,35-52,53-69
    # We use generic quartile-based zones
    for col in ['n1','n2','n3','n4','n5']:
        df[f'{col}_quad'] = pd.cut(df[col], bins=4, labels=[1,2,3,4]).astype(float)
    df['quad1_count'] = (df[['n1_quad','n2_quad','n3_quad','n4_quad','n5_quad']] == 1).sum(axis=1)
    df['quad4_count'] = (df[['n1_quad','n2_quad','n3_quad','n4_quad','n5_quad']] == 4).sum(axis=1)

    # ── hot/cold rolling frequency windows ──
    for col in ['n1','n2','n3','n4','n5']:
        for w in [10, 20, 50]:
            df[f'{col}_roll{w}'] = df[col].rolling(w).mean()

    # ── inter-draw deltas ──
    for col in ['n1','n2','n3','n4','n5']:
        df[f'{col}_delta'] = df[col].diff()
        df[f'{col}_lag1']  = df[col].shift(1)
        df[f'{col}_lag2']  = df[col].shift(2)

    # ── sum of consecutive gaps (spread shape) ──
    df['gap_sum']      = df[['gap_1_2','gap_2_3','gap_3_4','gap_4_5']].sum(axis=1)
    df['gap_std']      = df[['gap_1_2','gap_2_3','gap_3_4','gap_4_5']].std(axis=1)
    df['gap_max']      = df[['gap_1_2','gap_2_3','gap_3_4','gap_4_5']].max(axis=1)

    # ── spec ball features ──
    df[f'{spec_col}_lag1']  = df[spec_col].shift(1)
    df[f'{spec_col}_roll3'] = df[spec_col].rolling(3).mean()
    df[f'{spec_col}_delta'] = df[spec_col].diff()

    return df.dropna().reset_index(drop=True)

pb_feat = build_features(pb_raw, 'pb')
mm_feat = build_features(mm_raw, 'mb')
print(f"\n📊 pb_feat: {pb_feat.shape} | mm_feat: {mm_feat.shape}")

# ───────────────────────────────────────────────────────────────
#  3. APRIORI
# ───────────────────────────────────────────────────────────────
def run_apriori(df, label):
    print(f"\n=== {label} Apriori ===")
    transactions = [[f"n{i}={int(row[f'n{i}'])}" for i in range(1,6)] for _, row in df.iterrows()]
    te    = TransactionEncoder()
    te_df = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)
    freq  = apriori(te_df, min_support=0.015, use_colnames=True)
    rules = association_rules(freq, metric="confidence", min_threshold=0.3)
    print(f"✅ {label} Apriori: {len(freq)} itemsets, {len(rules)} rules")
    return freq, rules

pb_freq, pb_rules = run_apriori(pb_feat, "Powerball")
mm_freq, mm_rules = run_apriori(mm_feat, "MegaMillions")

# ───────────────────────────────────────────────────────────────
#  4. O-INFO  orders 2 → 24  (adaptive bins, beam=200)
# ───────────────────────────────────────────────────────────────
O_COLS = [
    'draw_sum','draw_std','draw_range','draw_mean','draw_median','draw_skew',
    'gap_sum','gap_std','gap_max','gap_1_2','gap_2_3','gap_3_4','gap_4_5',
    'even_count','odd_count','prime_count','fib_count','quad1_count','quad4_count',
    'n1_n2_sum','n2_n3_sum','n3_n4_sum','n4_n5_sum','n1_n3_sum','n1_n5_sum','n2_n4_sum',
    'n1_n2_diff','n2_n3_diff','n3_n4_diff','n4_n5_diff',
    'n1_mod3','n2_mod3','n3_mod3','n4_mod3','n5_mod3',
    'n1_mod5','n2_mod5','n3_mod5','n4_mod5','n5_mod5',
    'n1_mod7','n2_mod7','n3_mod7','n4_mod7','n5_mod7',
    'n1_mod11','n2_mod11','n3_mod11','n4_mod11','n5_mod11',
    'n1_mod13','n2_mod13','n3_mod13','n4_mod13','n5_mod13',
    'n1_digsum','n2_digsum','n3_digsum','n4_digsum','n5_digsum',
]

def o_info_tuple(df, cols, bins=None):
    n = len(cols)
    if bins is None:
        bins = max(3, 10 - n)
    data   = {c: pd.cut(df[c], bins=bins, labels=False).fillna(0).astype(int).values for c in cols}
    joint  = np.ravel_multi_index([data[c] for c in cols], [bins]*n, mode='clip')
    counts = np.bincount(joint, minlength=bins**n).astype(float) + 1e-9
    p      = counts / counts.sum()
    H_all  = -np.sum(p * np.log2(p))
    o      = (n - 2) * H_all
    for c in cols:
        rest = [r for r in cols if r != c]
        jr   = np.ravel_multi_index([data[r] for r in rest], [bins]*(n-1), mode='clip')
        cr   = np.bincount(jr, minlength=bins**(n-1)).astype(float) + 1e-9
        pr   = cr / cr.sum()
        o   -= -np.sum(pr * np.log2(pr))
    return float(o)

def run_oinfo(feat_df, label, max_order=24, top_k=2, beam=200, max_eval=3000):
    """
    beam    = top-N tuples kept as seeds for next order
    max_eval = max candidates actually scored per order (random sample if more)
               keeps each order under ~30s regardless of expansion
    """
    print(f"\n=== {label} O-Info (orders 2→{max_order}) ===")
    cols      = [c for c in O_COLS if c in feat_df.columns]
    prev_good = list(itertools.combinations(cols, 2))
    rng       = np.random.default_rng(42)

    for order in range(2, max_order + 1):
        if order == 2:
            candidates = prev_good
        else:
            seed = set()
            for tup in prev_good:
                for c in cols:
                    if c not in tup:
                        new = tuple(sorted(set(tup) | {c}))
                        if len(new) == order:
                            seed.add(new)
            candidates = list(seed)

        if not candidates:
            print(f"  Order {order}: 0 candidates — stopping"); break

        # ── subsample if too many — keeps runtime bounded at every order ──
        if len(candidates) > max_eval:
            idx        = rng.choice(len(candidates), max_eval, replace=False)
            candidates = [candidates[i] for i in idx]
            sampled    = True
        else:
            sampled    = False

        bins = max(3, 10 - order)
        tag  = f" [sampled {max_eval}/{len(candidates)+max_eval if sampled else len(candidates)}]" if sampled else ""
        print(f"  Order {order}: {len(candidates)} candidates [bins={bins}]{tag}", end=' | ', flush=True)

        scored = [(t, o_info_tuple(feat_df, list(t), bins=bins)) for t in candidates]
        synerg = sorted([(t,s) for t,s in scored if s < 0], key=lambda x: x[1])
        print(f"{len(synerg)} synergistic")

        for t, s in synerg[:top_k]:
            print(f"  🌀 {t} → O-info={s:.4f}")

        if not synerg:
            print(f"  Order {order}: no synergistic — stopping"); break

        prev_good = [r[0] for r in synerg[:beam]]

    print("✅ O-Info done")
run_oinfo(pb_feat, "Powerball")
run_oinfo(mm_feat, "MegaMillions")

# ───────────────────────────────────────────────────────────────
#  5. PHYSICS SIM
# ───────────────────────────────────────────────────────────────
def physics_sim(n_balls, n_draws=50_000, seed=0, label=""):
    print(f"\n⚙️  Physics Sim — {label} ({n_balls} balls, {n_draws:,} draws)")
    rng    = np.random.default_rng(seed)
    counts = np.zeros(n_balls + 1)
    mass   = rng.uniform(0.95, 1.05, n_balls + 1)
    vel    = rng.uniform(-1, 1,      n_balls + 1)
    for _ in tqdm(range(n_draws), desc=f"  {label}", ncols=80):
        vel   += rng.uniform(-0.1, 0.1, n_balls + 1)
        vel   *= 0.99
        counts[np.argsort(np.abs(vel) / mass)[-5:]] += 1
    top10 = list(np.argsort(counts[1:])[-10:][::-1] + 1)
    expected = n_draws * 5 / n_balls
    print(f"  ✅ Top-10: {top10} | max deviation: {(counts[1:].max()-expected)/expected*100:.2f}%")
    return top10

pb_phys_white = physics_sim(69, label="Powerball-White")
pb_phys_red   = physics_sim(26, label="Powerball-Red",  seed=1)
mm_phys_white = physics_sim(70, label="MegaMil-White",  seed=2)
mm_phys_gold  = physics_sim(25, label="MegaMil-Gold",   seed=3)

# ───────────────────────────────────────────────────────────────
#  6. DMD  (fixed: center data before SVD)
# ───────────────────────────────────────────────────────────────
def run_dmd(feat_df, label, n_modes=10):
    print(f"\n🌀 DMD — {label}")
    # Use only raw ball columns + draw stats — NOT the DMD cols themselves
    base_cols = ['n1','n2','n3','n4','n5','draw_sum','draw_std','draw_range',
                 'draw_mean','even_count','prime_count','gap_sum']
    num_cols  = [c for c in base_cols if c in feat_df.columns]
    X         = feat_df[num_cols].values.T.astype(float)
    # ── FIX: center each row so DMD finds oscillations not just means ──
    X         = X - X.mean(axis=1, keepdims=True)
    X1, X2    = X[:, :-1], X[:, 1:]
    U, s, Vh  = np.linalg.svd(X1, full_matrices=False)
    r         = min(n_modes, len(s))
    A_tilde   = U[:,:r].T @ X2 @ Vh[:r,:].T @ np.linalg.inv(np.diag(s[:r]))
    eigs      = np.linalg.eigvals(A_tilde)
    freqs     = np.angle(eigs) / (2 * np.pi)
    top5      = [round(f, 4) for f in sorted(freqs.real, key=abs)[:5]]
    print(f"  Top-5 freqs: {top5}")
    for i, eig in enumerate(eigs[:n_modes]):
        feat_df[f'dmd_re_{i}'] = np.real(eig)
        feat_df[f'dmd_im_{i}'] = np.imag(eig)
    print(f"  ✅ DMD cols: {n_modes}")
    return feat_df

pb_feat = run_dmd(pb_feat, "Powerball")
mm_feat = run_dmd(mm_feat, "MegaMillions")

# ───────────────────────────────────────────────────────────────
#  7. TDA
# ───────────────────────────────────────────────────────────────
def run_tda(feat_df, label, window=30):
    if not HAS_RIPSER:
        return feat_df
    print(f"🔺 TDA — {label}")
    num_cols  = ['n1','n2','n3','n4','n5','draw_sum','draw_std','draw_range','even_count','prime_count']
    num_cols  = [c for c in num_cols if c in feat_df.columns]
    h1_counts = []
    for i in range(len(feat_df)):
        start = max(0, i - window)
        patch = feat_df[num_cols].values[start:i+1]
        if len(patch) < 4:
            h1_counts.append(0); continue
        patch = (patch - patch.mean(0)) / (patch.std(0) + 1e-9)
        dgms  = ripser.ripser(patch, maxdim=1)['dgms']
        h1    = dgms[1] if len(dgms) > 1 else np.empty((0,2))
        h1_counts.append(len(h1[np.isfinite(h1).all(axis=1)]))
    feat_df['tda_h1_count'] = h1_counts
    print(f"  ✅ TDA cols added | avg H1 loops: {np.mean(h1_counts):.2f}")
    return feat_df

pb_feat = run_tda(pb_feat, "Powerball")
mm_feat = run_tda(mm_feat, "MegaMillions")
print(f"\n📊 Final pb_feat: {pb_feat.shape} | mm_feat: {mm_feat.shape}")

# ───────────────────────────────────────────────────────────────
#  8. ESN  — fixed: (N,1) state vector + NO-REPEAT enforcement
# ───────────────────────────────────────────────────────────────
class ESN:
    def __init__(self, N=300, spectral_radius=0.9, sparsity=0.1,
                 alpha=0.3, ridge_alpha=1e-4, seed=42):
        self.N = N; self.alpha = alpha
        self.ridge = Ridge(alpha=ridge_alpha)
        self.sx = StandardScaler(); self.sy = StandardScaler()
        rng = np.random.default_rng(seed)
        W   = rng.standard_normal((N, N)) * (rng.random((N, N)) < sparsity)
        ev  = np.max(np.abs(np.linalg.eigvals(W)))
        self.W_res = (W / ev) * spectral_radius if ev > 0 else W
        self.W_in  = rng.standard_normal((N, 1)) * 0.1

    def _states(self, inputs):
        x = np.zeros((self.N, 1))                          # ← (N,1) column vector
        S = np.zeros((len(inputs), self.N))
        for t, u in enumerate(inputs):
            x    = (1 - self.alpha) * x + self.alpha * np.tanh(
                   self.W_res @ x + self.W_in * float(u))  # ← scalar multiply, stays (N,1)
            S[t] = x.flatten()
        return S

    def fit(self, series, washout=50):
        s = self.sx.fit_transform(series.reshape(-1, 1)).flatten()
        S = self._states(s)
        self.ridge.fit(S[washout:-1],
                       self.sy.fit_transform(s[washout+1:].reshape(-1, 1)).flatten())
        return self

    def predict_next(self, series):
        s = self.sx.transform(series.reshape(-1, 1)).flatten()
        return float(self.sy.inverse_transform(
            self.ridge.predict(self._states(s)[-1:].reshape(1, -1)).reshape(-1, 1))[0, 0])

def pick_unique(raw_float, used, lo, hi):
    """Round to nearest int not already in used, searching outward."""
    base = int(np.clip(round(raw_float), lo, hi))
    for delta in range(0, hi - lo + 1):
        for candidate in [base + delta, base - delta]:
            if lo <= candidate <= hi and candidate not in used:
                return candidate
    return base  # fallback (shouldn't happen)

def run_esn(feat_df, spec_col, white_max, spec_max, game,
            extra_feats=None, extra_label=None):
    tag = f" (+ {extra_label} cascade)" if extra_label else ""
    print(f"\n🔮 ESN — {game}{tag}")
    df = feat_df.copy()
    if extra_feats is not None:
        n   = min(len(df), len(extra_feats))
        df  = df.iloc[-n:].copy().reset_index(drop=True)
        ext = extra_feats.iloc[-n:][['draw_sum','draw_std']].copy().reset_index(drop=True)
        df['ext_draw_sum'] = ext['draw_sum'].values
        df['ext_draw_std'] = ext['draw_std'].values
    preds = {}
    used  = set()
    for col in ['n1','n2','n3','n4','n5']:
        v        = df[col].values.astype(float)
        raw      = ESN(N=300, seed=42).fit(v).predict_next(v)
        chosen   = pick_unique(raw, used, 1, white_max)
        used.add(chosen)
        preds[col] = chosen
        print(f"  {col} → {chosen}  (raw={raw:.2f})")
    v = df[spec_col].values.astype(float)
    preds[spec_col] = int(np.clip(round(ESN(N=300,seed=42).fit(v).predict_next(v)), 1, spec_max))
    print(f"  {spec_col.upper()} → {preds[spec_col]}")
    return preds

pb_esn    = run_esn(pb_feat, 'pb', 69, 26, "Powerball")
mm_esn    = run_esn(mm_feat, 'mb', 70, 25, "MegaMillions")
pb_esn_mm = run_esn(pb_feat, 'pb', 69, 26, "Powerball",    extra_feats=mm_feat, extra_label="MegaMil")
mm_esn_pb = run_esn(mm_feat, 'mb', 70, 25, "MegaMillions", extra_feats=pb_feat, extra_label="Powerball")

# ───────────────────────────────────────────────────────────────
#  9. SEGMENT ANALYSIS  (DOW → Month → DOW×Month)
# ───────────────────────────────────────────────────────────────
def segment_accuracy(df, group_cols, white_max=70):
    rows = []
    for name, grp in df.groupby(group_cols):
        grp = grp.reset_index(drop=True)
        if len(grp) < 10: continue
        hits, total = 0, 0
        for i in range(max(10, len(grp)//3), len(grp)):
            train = grp.iloc[:i]; row = grp.iloc[i]
            preds = {}; used = set()
            for col in ['n1','n2','n3','n4','n5']:
                v = train[col].values.astype(float)
                if len(v) < 12: continue
                try:
                    raw    = ESN(N=100,seed=42).fit(v).predict_next(v)
                    chosen = pick_unique(raw, used, 1, white_max)
                    used.add(chosen); preds[col] = chosen
                except: pass
            if not preds: continue
            hits  += len({int(row[c]) for c in ['n1','n2','n3','n4','n5']} & set(preds.values()))
            total += 5
        if total > 0:
            r = {'accuracy': round(hits/total, 4), 'n_draws': len(grp)}
            if isinstance(name, tuple):
                for k, v in zip(group_cols, name): r[k] = v
            else:
                r[group_cols[0]] = name
            rows.append(r)
    return pd.DataFrame(rows).sort_values('accuracy', ascending=False)

print("\n=== SEGMENT ANALYSIS — Powerball ===")
for gc, lbl in [(['dow_name'],"DOW"), (['month'],"Month"), (['dow_name','month'],"DOW×Month top-10")]:
    seg = segment_accuracy(pb_feat, gc, white_max=69)
    print(f"\n{lbl}:"); print(seg.head(10).to_string(index=False))

print("\n=== SEGMENT ANALYSIS — MegaMillions ===")
for gc, lbl in [(['dow_name'],"DOW"), (['dow_name','month'],"DOW×Month top-10")]:
    seg = segment_accuracy(mm_feat, gc, white_max=70)
    print(f"\n{lbl}:"); print(seg.head(10).to_string(index=False))

# ───────────────────────────────────────────────────────────────
# 10. SQLITE TRACKER + DRAW SCHEDULE
# ───────────────────────────────────────────────────────────────
DB_PATH = Path("/kaggle/working/lottery_tracker.db")

def init_db(path=DB_PATH):
    con = sqlite3.connect(path)
    con.executescript("""
        CREATE TABLE IF NOT EXISTS predictions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            game TEXT, draw_date TEXT,
            pred_n1 INT, pred_n2 INT, pred_n3 INT,
            pred_n4 INT, pred_n5 INT, pred_spec INT,
            method TEXT, created_at TEXT DEFAULT (datetime('now'))
        );
        CREATE TABLE IF NOT EXISTS weekly_scores (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            game TEXT, week_start TEXT, draw_date TEXT,
            hits_white INT, hits_spec INT, method TEXT
        );
    """)
    con.commit(); return con

def save_pred(con, game, draw_date, preds, spec_col, method):
    con.execute("""INSERT INTO predictions
        (game,draw_date,pred_n1,pred_n2,pred_n3,pred_n4,pred_n5,pred_spec,method)
        VALUES(?,?,?,?,?,?,?,?,?)""",
        (game, str(draw_date),
         preds.get('n1'), preds.get('n2'), preds.get('n3'),
         preds.get('n4'), preds.get('n5'), preds.get(spec_col), method))
    con.commit()
    print(f"  💾 {method} → {game} {draw_date}")

def score_week(con, game, spec_col, raw_df):
    week_start = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')
    rows = con.execute(
        "SELECT draw_date,pred_n1,pred_n2,pred_n3,pred_n4,pred_n5,pred_spec,method "
        "FROM predictions WHERE game=? AND draw_date>=?", (game, week_start)).fetchall()
    if not rows:
        print(f"  ℹ️  No predictions on record for {game} this week"); return
    print(f"\n📋 Week Score — {game} (since {week_start})")
    for draw_date,p1,p2,p3,p4,p5,ps,method in rows:
        actual = raw_df[raw_df['draw_date'].astype(str).str[:10] == draw_date[:10]]
        if actual.empty:
            print(f"  {draw_date} [{method}]: result not available yet"); continue
        a  = actual.iloc[-1]
        hw = len({int(a['n1']),int(a['n2']),int(a['n3']),int(a['n4']),int(a['n5'])} & {p1,p2,p3,p4,p5})
        hs = 1 if int(a.get(spec_col, -1)) == ps else 0
        print(f"  {draw_date} [{method}]: {hw}/5 white  spec={'✅' if hs else '❌'}")
        con.execute("""INSERT OR REPLACE INTO weekly_scores
            (game,week_start,draw_date,hits_white,hits_spec,method) VALUES(?,?,?,?,?,?)""",
            (game, week_start, draw_date, hw, hs, method))
    con.commit()

PB_DAYS = {0, 2, 5}
MM_DAYS = {1, 4}

def next_draw(today, weekdays):
    for offset in range(1, 8):
        d = today + timedelta(days=offset)
        if d.weekday() in weekdays:
            return d.strftime('%Y-%m-%d')

now     = datetime.now()
next_pb = next_draw(now, PB_DAYS)
next_mm = next_draw(now, MM_DAYS)

print(f"\n📅 DRAW SCHEDULE")
print(f"  Powerball     → Mon / Wed / Sat   10:59 pm ET  | Next: {next_pb}")
print(f"  Mega Millions → Tue / Fri         11:00 pm ET  | Next: {next_mm}")

con = init_db()
save_pred(con, 'Powerball',    next_pb, pb_esn,    'pb', 'ESN')
save_pred(con, 'Powerball',    next_pb, pb_esn_mm, 'pb', 'ESN+MM_cascade')
save_pred(con, 'MegaMillions', next_mm, mm_esn,    'mb', 'ESN')
save_pred(con, 'MegaMillions', next_mm, mm_esn_pb, 'mb', 'ESN+PB_cascade')

score_week(con, 'Powerball',    'pb', pb_raw)
score_week(con, 'MegaMillions', 'mb', mm_raw)

# ───────────────────────────────────────────────────────────────
# 11. FINAL SUMMARY
# ───────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL PREDICTIONS")
print("="*60)
for game, preds, spec, nxt in [
    ("Powerball (standalone)",   pb_esn,    'pb', next_pb),
    ("Powerball (+MM cascade)",  pb_esn_mm, 'pb', next_pb),
    ("MegaMil   (standalone)",   mm_esn,    'mb', next_mm),
    ("MegaMil   (+PB cascade)",  mm_esn_pb, 'mb', next_mm),
]:
    nums = ' '.join(str(preds.get(f'n{i}','?')) for i in range(1,6))
    print(f"  {game:<30} {nxt}  [{nums}]  {spec.upper()}:{preds.get(spec,'?')}")

print(f"\n✅ DB saved → {DB_PATH}")
print("✅ All done!")

✅ Device: cuda
✅ ripser available — running TDA
🌐 Fetching Powerball...
✅ Powerball: 1933 draws | Latest: 2026-04-27 | 18-31-33-36-62 PB:3
🌐 Fetching Mega Millions...
✅ Mega Millions: 2496 draws | Latest: 2026-04-24 | 7-16-32-35-40 MB:12

📊 pb_feat: (1674, 131) | mm_feat: (1483, 131)

=== Powerball Apriori ===
✅ Powerball Apriori: 142 itemsets, 0 rules

=== MegaMillions Apriori ===
✅ MegaMillions Apriori: 150 itemsets, 0 rules

=== Powerball O-Info (orders 2→24) ===
  Order 2: 1770 candidates [bins=8] | 1770 synergistic
  🌀 ('n1_mod11', 'n2_mod13') → O-info=-5.8601
  🌀 ('n2_mod13', 'n5_mod13') → O-info=-5.8598
  Order 3: 3000 candidates [bins=7] [sampled 3000/6000] | 3000 synergistic
  🌀 ('draw_median', 'n3_mod11', 'n3_mod7') → O-info=-9.7142
  🌀 ('n4_digsum', 'n4_mod11', 'n4_mod7') → O-info=-9.6483
  Order 4: 3000 candidates [bins=6] [sampled 3000/6000] | 3000 synergistic
  🌀 ('n1_mod13', 'n4_digsum', 'n4_mod11', 'n4_mod5') → O-info=-11.6645
  🌀 ('n1_mod7', 'n4_digsum', 'n4_mod11', 'n

KeyboardInterrupt: 